In [ ]:
# Install if needed
!pip install aif360 scikit-learn pandas numpy

import numpy as np
import pandas as pd

from aif360.datasets import AdultDataset, CompasDataset, GermanDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
def load_adult():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

    columns = [
        "age", "workclass", "fnlwgt", "education", "education-num",
        "marital-status", "occupation", "relationship", "race", "sex",
        "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"
    ]

    df = pd.read_csv(url, names=columns, na_values=" ?", skipinitialspace=True)

    df['label'] = df['income'].apply(lambda x: 1 if x == '>50K' else 0)
    df = df.drop(columns=['income'])

    df = pd.get_dummies(df, drop_first=False)

    return df

In [ ]:
def load_compas():
    url = "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv"

    df = pd.read_csv(url)

    df = df[
        (df['days_b_screening_arrest'] <= 30) &
        (df['days_b_screening_arrest'] >= -30) &
        (df['is_recid'] != -1) &
        (df['c_charge_degree'] != 'O') &
        (df['score_text'] != 'N/A')
    ]

    df = df[['age', 'race', 'sex', 'priors_count', 'c_charge_degree', 'two_year_recid']]

    df['label'] = df['two_year_recid']
    df = df.drop(columns=['two_year_recid'])

    df = pd.get_dummies(df, drop_first=False)

    return df

In [ ]:
def load_german():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"

    columns = [
        "checking", "duration", "credit_history", "purpose", "amount",
        "savings", "employment", "installment_rate", "sex", "other_debtors",
        "residence", "property", "age", "other_installment",
        "housing", "existing_credits", "job", "num_people",
        "telephone", "foreign_worker", "label"
    ]

    df = pd.read_csv(url, sep=' ', names=columns)

    df['label'] = df['label'].apply(lambda x: 1 if x == 1 else 0)

    # convert sex to binary
    df['sex'] = df['sex'].apply(lambda x: 1 if x in ['A91','A93','A94'] else 0)

    df = pd.get_dummies(df, drop_first=False)

    return df

In [ ]:
def get_protected(df, dataset_name):
    if dataset_name == "adult":
        return pd.DataFrame({
            "sex": df['sex_Male'],
            "race": df['race_White']
        })

    elif dataset_name == "compas":
        return pd.DataFrame({
            "sex": df['sex_Male'],
            "race": df['race_African-American']
        })

    elif dataset_name == "german":
        return pd.DataFrame({
            "sex": df['sex']
        })

In [ ]:
def prepare_data(df, protected_df):
    X = df.drop(columns=['label'])
    y = df['label']

    X_train, X_test, y_train, y_test, prot_train, prot_test = train_test_split(
        X, y, protected_df, test_size=0.2, random_state=42
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    return X_train, X_test, y_train, y_test, prot_train, prot_test

In [ ]:
def save_data(name, X_train, X_test, y_train, y_test, prot_train, prot_test):
    np.save(f"{name}_X_train.npy", X_train)
    np.save(f"{name}_X_test.npy", X_test)
    np.save(f"{name}_y_train.npy", y_train)
    np.save(f"{name}_y_test.npy", y_test)
    np.save(f"{name}_prot_train.npy", prot_train)
    np.save(f"{name}_prot_test.npy", prot_test)

    print(f"{name} saved!")

In [ ]:
datasets = {
    "adult": load_adult,
    "compas": load_compas,
    "german": load_german
}

all_data = {}

for name, loader in datasets.items():
    print(f"\nProcessing {name}...")

    df = loader()

    protected = get_protected(df, name)

    X_train, X_test, y_train, y_test, prot_train, prot_test = prepare_data(df, protected)

    save_data(name, X_train, X_test, y_train, y_test, prot_train, prot_test)

    all_data[name] = (X_train, X_test, y_train, y_test, prot_train, prot_test)


Processing adult...
adult saved!

Processing compas...
compas saved!

Processing german...
german saved!


In [ ]:
def load_data(name):
    X_train = np.load(f"{name}_X_train.npy")
    X_test = np.load(f"{name}_X_test.npy")
    y_train = np.load(f"{name}_y_train.npy")
    y_test = np.load(f"{name}_y_test.npy")

    prot_train = np.load(f"{name}_prot_train.npy")
    prot_test = np.load(f"{name}_prot_test.npy")

    #convert back to DataFrame with column names
    if name == "adult":
        cols = ["sex", "race"]
    elif name == "compas":
        cols = ["sex", "race"]
    elif name == "german":
        cols = ["sex"]

    prot_train = pd.DataFrame(prot_train, columns=cols)
    prot_test = pd.DataFrame(prot_test, columns=cols)

    return X_train, X_test, y_train, y_test, prot_train, prot_test

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

def train_models(X_train, y_train):
    models = {}

    # Logistic Regression
    lr = LogisticRegression(max_iter=1000)
    lr.fit(X_train, y_train)
    models["LogisticRegression"] = lr

    # Random Forest
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    models["RandomForest"] = rf

    return models

In [ ]:
from sklearn.metrics import accuracy_score

def evaluate_models(models, X_test, y_test):
    results = {}

    for name, model in models.items():
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)

        results[name] = {
            "accuracy": acc,
            "predictions": y_pred
        }

        print(f"{name} Accuracy: {acc:.4f}")

    return results

In [ ]:
dataset_names = ["adult", "compas", "german"]

all_results = {}

for name in dataset_names:
    print(f"\n===== {name.upper()} =====")

    X_train, X_test, y_train, y_test, prot_train, prot_test = load_data(name)

    models = train_models(X_train, y_train)

    results = evaluate_models(models, X_test, y_test)

    all_results[name] = {
        "models": models,
        "results": results,
        "protected": prot_test,
        "y_test": y_test
    }


===== ADULT =====
LogisticRegression Accuracy: 0.8572
RandomForest Accuracy: 0.8595

===== COMPAS =====
LogisticRegression Accuracy: 0.6607
RandomForest Accuracy: 0.6186

===== GERMAN =====
LogisticRegression Accuracy: 0.8050
RandomForest Accuracy: 0.8000


In [ ]:
def demographic_parity(y_pred, protected_attr):
    group_1 = y_pred[protected_attr == 1]
    group_0 = y_pred[protected_attr == 0]

    if len(group_1) == 0 or len(group_0) == 0:
        return 0

    return abs(group_1.mean() - group_0.mean())


def equal_opportunity(y_true, y_pred, protected_attr):
    # TPR = P(Ŷ = 1 | Y = 1)

    mask_1 = (protected_attr == 1) & (y_true == 1)
    mask_0 = (protected_attr == 0) & (y_true == 1)

    if mask_1.sum() == 0 or mask_0.sum() == 0:
        return 0

    tpr_1 = y_pred[mask_1].mean()
    tpr_0 = y_pred[mask_0].mean()

    return abs(tpr_1 - tpr_0)

In [ ]:
def evaluate_fairness(results, y_test, prot_test):
    fairness_results = {}

    for model_name, res in results.items():
        y_pred = res["predictions"]

        fairness = {}

        for attr in prot_test.columns:
            protected_attr = prot_test[attr].values

            dp = demographic_parity(y_pred, protected_attr)
            eo = equal_opportunity(y_test, y_pred, protected_attr)

            fairness[attr] = {
                "demographic_parity": dp,
                "equal_opportunity": eo
            }

            print(f"{model_name} | {attr}")
            print(f"  DP diff: {dp:.4f}")
            print(f"  EO diff: {eo:.4f}")

        fairness_results[model_name] = fairness

    return fairness_results

In [ ]:
for name in dataset_names:
    print(f"\n===== FAIRNESS: {name.upper()} =====")

    data = all_results[name]

    fairness = evaluate_fairness(
        data["results"],
        data["y_test"],
        data["protected"]
    )

    all_results[name]["fairness"] = fairness


===== FAIRNESS: ADULT =====
LogisticRegression | sex
  DP diff: 0.1893
  EO diff: 0.1914
LogisticRegression | race
  DP diff: 0.0791
  EO diff: 0.0058
RandomForest | sex
  DP diff: 0.1934
  EO diff: 0.1565
RandomForest | race
  DP diff: 0.0860
  EO diff: 0.0117

===== FAIRNESS: COMPAS =====
LogisticRegression | sex
  DP diff: 0.3765
  EO diff: 0.4775
LogisticRegression | race
  DP diff: 0.3071
  EO diff: 0.2876
RandomForest | sex
  DP diff: 0.2198
  EO diff: 0.2209
RandomForest | race
  DP diff: 0.2170
  EO diff: 0.2237

===== FAIRNESS: GERMAN =====
LogisticRegression | sex
  DP diff: 0.0139
  EO diff: 0.0505
RandomForest | sex
  DP diff: 0.0337
  EO diff: 0.0181


In [ ]:
# Expanded Research Phase (Final)
#
# This section strengthens the project in five ways:
# 1. Broader data coverage with a fourth dataset from a different domain.
# 2. More model families including linear, bagging, and boosting methods.
# 3. Richer predictive evaluation using accuracy, balanced accuracy, F1, and ROC-AUC.
# 4. Stronger fairness analysis using demographic parity, equal opportunity,
#    equalized odds proxy, and predictive parity proxy.
# 5. A socially responsible utility score that combines predictive quality,
#    fairness, and compute cost as an energy proxy.
#
# This section is designed to satisfy the final report expectation of multiple
# algorithms and multiple evaluation methods, while supporting bonus-credit
# readiness through comprehensive comparison.



In [ ]:
import time
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

warnings.filterwarnings("ignore")

In [ ]:
def load_titanic():
    # Different domain dataset for broader comparison
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    df = pd.read_csv(url)

    # Label: survival
    df['label'] = df['Survived'].astype(int)

    # Protected attributes: sex and age group
    protected = pd.DataFrame({
        'sex_male': (df['Sex'] == 'male').astype(int),
        'age_group_40_plus': (df['Age'].fillna(df['Age'].median()) >= 40).astype(int)
    })

    drop_cols = ['Survived', 'Name', 'Ticket', 'Cabin', 'PassengerId']
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])

    # Fill numeric missing values before one hot encoding
    for col in df.select_dtypes(include=[np.number]).columns:
        df[col] = df[col].fillna(df[col].median())

    for col in df.select_dtypes(exclude=[np.number]).columns:
        df[col] = df[col].fillna('Unknown')

    df = pd.get_dummies(df, drop_first=False)
    return df, protected


def build_dataset_bundle():
    bundle = {}

    # Reuse the three original datasets
    adult_df = load_adult()
    bundle['adult'] = (adult_df, get_protected(adult_df, 'adult'))

    compas_df = load_compas()
    bundle['compas'] = (compas_df, get_protected(compas_df, 'compas'))

    german_df = load_german()
    bundle['german'] = (german_df, get_protected(german_df, 'german'))

    titanic_df, titanic_protected = load_titanic()
    bundle['titanic'] = (titanic_df, titanic_protected)

    return bundle

In [ ]:
def prepare_split(df, protected_df, test_size=0.2, val_size=0.2, random_state=42):
    X = df.drop(columns=['label'])
    y = df['label']

    X_train, X_test, y_train, y_test, prot_train, prot_test = train_test_split(
        X, y, protected_df, test_size=test_size, random_state=random_state, stratify=y
    )

    # Validation split for lightweight tuning/comparison
    X_train, X_val, y_train, y_val, prot_train, prot_val = train_test_split(
        X_train, y_train, prot_train, test_size=val_size, random_state=random_state, stratify=y_train
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    return {
        'X_train': X_train_scaled,
        'X_val': X_val_scaled,
        'X_test': X_test_scaled,
        'y_train': y_train.values,
        'y_val': y_val.values,
        'y_test': y_test.values,
        'prot_val': prot_val.reset_index(drop=True),
        'prot_test': prot_test.reset_index(drop=True)
    }


def get_model_suite(random_state=42):
    return {
        'LogisticRegression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=random_state),
        'RandomForest': RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight='balanced_subsample',
            random_state=random_state,
            n_jobs=-1
        ),
        'GradientBoosting': GradientBoostingClassifier(random_state=random_state)
    }

In [ ]:
def fairness_metrics(y_true, y_pred, protected_attr):
    # Demographic parity: difference in positive prediction rates
    group_1 = (protected_attr == 1)
    group_0 = (protected_attr == 0)

    dp_diff = abs(y_pred[group_1].mean() - y_pred[group_0].mean()) if group_1.any() and group_0.any() else np.nan

    # Equal opportunity: difference in TPR (among positives)
    pos_1 = group_1 & (y_true == 1)
    pos_0 = group_0 & (y_true == 1)
    tpr_1 = y_pred[pos_1].mean() if pos_1.any() else np.nan
    tpr_0 = y_pred[pos_0].mean() if pos_0.any() else np.nan
    eo_diff = abs(tpr_1 - tpr_0) if not np.isnan(tpr_1) and not np.isnan(tpr_0) else np.nan

    # Equalized odds proxy: average of |TPR diff| and |FPR diff|
    neg_1 = group_1 & (y_true == 0)
    neg_0 = group_0 & (y_true == 0)
    fpr_1 = y_pred[neg_1].mean() if neg_1.any() else np.nan
    fpr_0 = y_pred[neg_0].mean() if neg_0.any() else np.nan
    fpr_diff = abs(fpr_1 - fpr_0) if not np.isnan(fpr_1) and not np.isnan(fpr_0) else np.nan

    eq_odds_diff = np.nanmean([eo_diff, fpr_diff])

    # Predictive parity proxy: precision difference across groups
    pred_pos_1 = group_1 & (y_pred == 1)
    pred_pos_0 = group_0 & (y_pred == 1)
    ppv_1 = y_true[pred_pos_1].mean() if pred_pos_1.any() else np.nan
    ppv_0 = y_true[pred_pos_0].mean() if pred_pos_0.any() else np.nan
    pp_diff = abs(ppv_1 - ppv_0) if not np.isnan(ppv_1) and not np.isnan(ppv_0) else np.nan

    return {
        'dp_diff': dp_diff,
        'eo_diff': eo_diff,
        'eq_odds_diff': eq_odds_diff,
        'pp_diff': pp_diff
    }


def evaluate_predictions(y_true, y_pred, y_prob):
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred)
    }
    try:
        metrics['roc_auc'] = roc_auc_score(y_true, y_prob)
    except Exception:
        metrics['roc_auc'] = np.nan
    return metrics


def run_research_experiments(dataset_bundle):
    rows = []

    for dataset_name, (df, protected_df) in dataset_bundle.items():
        split = prepare_split(df, protected_df)

        for model_name, model in get_model_suite().items():
            start = time.perf_counter()
            model.fit(split['X_train'], split['y_train'])
            train_time_sec = time.perf_counter() - start

            y_pred_test = model.predict(split['X_test'])
            y_prob_test = model.predict_proba(split['X_test'])[:, 1] if hasattr(model, 'predict_proba') else y_pred_test

            perf = evaluate_predictions(split['y_test'], y_pred_test, y_prob_test)

            # Aggregate fairness over all protected attributes in this dataset
            fairness_per_attr = []
            for attr in split['prot_test'].columns:
                f = fairness_metrics(split['y_test'], y_pred_test, split['prot_test'][attr].values)
                f['attribute'] = attr
                fairness_per_attr.append(f)

            fairness_df = pd.DataFrame(fairness_per_attr)

            # Socially responsible utility: reward performance, penalize unfairness and compute time.
            # This helps make explicit performance-fairness-resource tradeoffs.
            avg_fairness_penalty = fairness_df[['dp_diff', 'eo_diff', 'eq_odds_diff', 'pp_diff']].mean().mean()
            utility = (0.6 * perf['balanced_accuracy']) + (0.4 * perf['f1']) - (0.35 * avg_fairness_penalty) - (0.02 * np.log1p(train_time_sec))

            rows.append({
                'dataset': dataset_name,
                'model': model_name,
                'accuracy': perf['accuracy'],
                'balanced_accuracy': perf['balanced_accuracy'],
                'f1': perf['f1'],
                'roc_auc': perf['roc_auc'],
                'train_time_sec': train_time_sec,
                'avg_dp_diff': fairness_df['dp_diff'].mean(),
                'avg_eo_diff': fairness_df['eo_diff'].mean(),
                'avg_eq_odds_diff': fairness_df['eq_odds_diff'].mean(),
                'avg_pp_diff': fairness_df['pp_diff'].mean(),
                'avg_fairness_penalty': avg_fairness_penalty,
                'srai_utility': utility
            })

    return pd.DataFrame(rows)

In [ ]:
dataset_bundle = build_dataset_bundle()
research_results = run_research_experiments(dataset_bundle)

# Display by dataset and SRAI utility
research_results_sorted = research_results.sort_values(['dataset', 'srai_utility'], ascending=[True, False])
research_results_sorted.round(4)

,dataset,model,accuracy,balanced_accuracy,f1,roc_auc,train_time_sec,avg_dp_diff,avg_eo_diff,avg_eq_odds_diff,avg_pp_diff,avg_fairness_penalty,srai_utility
1,adult,RandomForest,0.8362,0.8311,0.7071,0.9183,0.7369,0.2193,0.1045,0.1189,0.0241,0.1167,0.7296
0,adult,LogisticRegression,0.8111,0.8255,0.6851,0.9080,0.1721,0.2397,0.1115,0.1352,0.0155,0.1255,0.7223
2,adult,GradientBoosting,0.8721,0.7882,0.7022,0.9253,2.2998,0.1289,0.0616,0.0538,0.0588,0.0758,0.7034
5,compas,GradientBoosting,0.7020,0.6945,0.6509,0.7417,0.1484,0.2841,0.3020,0.2455,0.0882,0.2300,0.5938
3,compas,LogisticRegression,0.6834,0.6827,0.6597,0.7303,0.0022,0.3728,0.3769,0.3361,0.1404,0.3065,0.5661
4,compas,RandomForest,0.6632,0.6594,0.6252,0.7092,0.3852,0.2794,0.2760,0.2451,0.0953,0.2239,0.5608
7,german,RandomForest,0.7350,0.6821,0.8114,0.7956,0.3043,0.0548,0.0550,0.0275,0.0468,0.0460,0.7124
8,german,GradientBoosting,0.7450,0.6798,0.8223,0.7805,0.1080,0.1214,0.1300,0.0900,0.0456,0.0968,0.7009
6,german,LogisticRegression,0.6600,0.6571,0.7323,0.7468,0.0018,0.0762,0.1250,0.1000,0.1014,0.1007,0.6519
11,titanic,GradientBoosting,0.8156,0.7906,0.7402,0.8098,0.0571,0.3158,0.2750,0.2129,0.2425,0.2615,0.6778


In [ ]:
summary = (
    research_results
    .groupby('model')[['balanced_accuracy', 'f1', 'roc_auc', 'avg_fairness_penalty', 'train_time_sec', 'srai_utility']]
    .mean()
    .sort_values('srai_utility', ascending=False)
)

print('Average performance across all datasets:')
display(summary.round(4))

best_per_dataset = (
    research_results
    .sort_values('srai_utility', ascending=False)
    .groupby('dataset')
    .head(1)
    [['dataset', 'model', 'srai_utility', 'balanced_accuracy', 'f1', 'avg_fairness_penalty', 'train_time_sec']]
)

print('Best method by dataset under SRAI utility:')
display(best_per_dataset.round(4))

Average performance across all datasets:


,balanced_accuracy,f1,roc_auc,avg_fairness_penalty,train_time_sec,srai_utility
model,,,,,,
GradientBoosting,0.7382,0.7289,0.8143,0.1660,0.6533,0.6689
RandomForest,0.7417,0.7225,0.8116,0.1796,0.4344,0.6641
LogisticRegression,0.7380,0.7042,0.8054,0.2226,0.0444,0.6458


Best method by dataset under SRAI utility:


,dataset,model,srai_utility,balanced_accuracy,f1,avg_fairness_penalty,train_time_sec
1,adult,RandomForest,0.7296,0.8311,0.7071,0.1167,0.7369
7,german,RandomForest,0.7124,0.6821,0.8114,0.0460,0.3043
11,titanic,GradientBoosting,0.6778,0.7906,0.7402,0.2615,0.0571
5,compas,GradientBoosting,0.5938,0.6945,0.6509,0.2300,0.1484


### Notes for Final Report / Slides

- We now compare **3 model families** across **4 datasets** (adult, compas, german, titanic), improving breadth and thoroughness.
- Beyond plain accuracy, we report **balanced accuracy, F1, and ROC-AUC**, which is better for imbalanced social datasets.
- We include multiple fairness criteria (**DP, EO, Equalized Odds proxy, Predictive Parity proxy**) and aggregate them for cross-dataset comparison.
- We explicitly account for computation time as an energy proxy in `srai_utility`, linking responsible AI to both fairness and resource efficiency.
- This supports a stronger final narrative: there is no universally best model; the preferred method depends on the performance-fairness-resource tradeoff and domain context.


## Extensive Comparison

To directly address the midterm feedback, we run repeated experiments across datasets and methods, and we explicitly analyze the relationship between **compute/energy proxy** and **fairness outcomes**.

Key additions:
- Multi-seed robustness analysis (not one split only)
- Broader method comparison including a low-cost baseline
- Correlation analysis between compute time and fairness disparity
- Pareto-style ranking over performance, fairness, and compute


In [ ]:
from sklearn.dummy import DummyClassifier


def get_extended_model_suite(random_state=42):
    return {
        'DummyMostFrequent': DummyClassifier(strategy='most_frequent'),
        'LogisticRegression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=random_state),
        'RandomForest': RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight='balanced_subsample',
            random_state=random_state,
            n_jobs=-1
        ),
        'GradientBoosting': GradientBoostingClassifier(random_state=random_state)
    }


def run_repeated_experiments(dataset_bundle, seeds=(7, 21, 42, 84, 126)):
    rows = []

    for seed in seeds:
        for dataset_name, (df, protected_df) in dataset_bundle.items():
            split = prepare_split(df, protected_df, random_state=seed)

            for model_name, model in get_extended_model_suite(random_state=seed).items():
                start = time.perf_counter()
                model.fit(split['X_train'], split['y_train'])
                train_time_sec = time.perf_counter() - start

                y_pred_test = model.predict(split['X_test'])
                y_prob_test = model.predict_proba(split['X_test'])[:, 1] if hasattr(model, 'predict_proba') else y_pred_test

                perf = evaluate_predictions(split['y_test'], y_pred_test, y_prob_test)

                fairness_per_attr = []
                for attr in split['prot_test'].columns:
                    f = fairness_metrics(split['y_test'], y_pred_test, split['prot_test'][attr].values)
                    f['attribute'] = attr
                    fairness_per_attr.append(f)

                fairness_df = pd.DataFrame(fairness_per_attr)
                avg_fairness_penalty = fairness_df[['dp_diff', 'eo_diff', 'eq_odds_diff', 'pp_diff']].mean().mean()

                utility = (0.6 * perf['balanced_accuracy']) + (0.4 * perf['f1']) - (0.35 * avg_fairness_penalty) - (0.02 * np.log1p(train_time_sec))

                rows.append({
                    'seed': seed,
                    'dataset': dataset_name,
                    'model': model_name,
                    'accuracy': perf['accuracy'],
                    'balanced_accuracy': perf['balanced_accuracy'],
                    'f1': perf['f1'],
                    'roc_auc': perf['roc_auc'],
                    'train_time_sec': train_time_sec,
                    'avg_dp_diff': fairness_df['dp_diff'].mean(),
                    'avg_eo_diff': fairness_df['eo_diff'].mean(),
                    'avg_eq_odds_diff': fairness_df['eq_odds_diff'].mean(),
                    'avg_pp_diff': fairness_df['pp_diff'].mean(),
                    'avg_fairness_penalty': avg_fairness_penalty,
                    'srai_utility': utility
                })

    return pd.DataFrame(rows)

In [ ]:
extensive_results = run_repeated_experiments(dataset_bundle)

method_summary = (
    extensive_results
    .groupby('model')[[
        'balanced_accuracy', 'f1', 'roc_auc',
        'avg_fairness_penalty', 'train_time_sec', 'srai_utility'
    ]]
    .agg(['mean', 'std'])
)

# Flatten columns for readability
method_summary.columns = [f"{a}_{b}" for a, b in method_summary.columns]
method_summary = method_summary.sort_values('srai_utility_mean', ascending=False)

print('Comprehensive comparison across methods (mean ± std over seeds x datasets):')
display(method_summary.round(4))

# Per-dataset top method under SRAI utility (mean over seeds)
per_dataset = (
    extensive_results
    .groupby(['dataset', 'model'])[['srai_utility', 'balanced_accuracy', 'avg_fairness_penalty', 'train_time_sec']]
    .mean()
    .reset_index()
)

best_per_dataset = (
    per_dataset
    .sort_values(['dataset', 'srai_utility'], ascending=[True, False])
    .groupby('dataset')
    .head(1)
)

print('Best method per dataset (mean over seeds):')
display(best_per_dataset.round(4))

Comprehensive comparison across methods (mean ± std over seeds x datasets):


,balanced_accuracy_mean,balanced_accuracy_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,avg_fairness_penalty_mean,avg_fairness_penalty_std,train_time_sec_mean,train_time_sec_std,srai_utility_mean,srai_utility_std
model,,,,,,,,,,,,
RandomForest,0.7411,0.0800,0.7258,0.0820,0.8136,0.0842,0.1669,0.0993,0.4040,0.1933,0.6700,0.0676
GradientBoosting,0.7291,0.0613,0.7242,0.0756,0.8184,0.0738,0.1594,0.0995,0.6573,0.9821,0.6639,0.0465
LogisticRegression,0.7405,0.0646,0.7049,0.0504,0.8128,0.0714,0.2068,0.1182,0.0660,0.1155,0.6527,0.0652
DummyMostFrequent,0.5000,0.0000,0.2059,0.3659,0.5000,0.0000,0.0049,0.0094,0.0002,0.0002,0.3806,0.1433


Best method per dataset (mean over seeds):


,dataset,model,srai_utility,balanced_accuracy,avg_fairness_penalty,train_time_sec
3,adult,RandomForest,0.7290,0.8303,0.1206,0.7229
5,compas,GradientBoosting,0.5939,0.6792,0.1853,0.1494
11,german,RandomForest,0.7192,0.6919,0.0642,0.2690
13,titanic,GradientBoosting,0.6668,0.7933,0.3014,0.0572


In [ ]:
\
# We use train_time_sec as an energy/computation proxy.

energy_fairness_corr = extensive_results[['train_time_sec', 'avg_fairness_penalty']].corr().iloc[0, 1]
energy_perf_corr = extensive_results[['train_time_sec', 'balanced_accuracy']].corr().iloc[0, 1]

print(f'Correlation(train_time_sec, fairness_penalty): {energy_fairness_corr:.4f}')
print(f'Correlation(train_time_sec, balanced_accuracy): {energy_perf_corr:.4f}')

# Ranking methods on each axis to reveal tradeoffs clearly
rank_table = (
    extensive_results
    .groupby('model')[['balanced_accuracy', 'avg_fairness_penalty', 'train_time_sec', 'srai_utility']]
    .mean()
    .assign(
        perf_rank=lambda d: d['balanced_accuracy'].rank(ascending=False, method='min'),
        fairness_rank=lambda d: d['avg_fairness_penalty'].rank(ascending=True, method='min'),
        energy_rank=lambda d: d['train_time_sec'].rank(ascending=True, method='min'),
        overall_rank=lambda d: d['srai_utility'].rank(ascending=False, method='min')
    )
    .sort_values('overall_rank')
)

print('Performance–Fairness–Energy ranking:')
display(rank_table.round(4))

Correlation(train_time_sec, fairness_penalty): -0.0912
Correlation(train_time_sec, balanced_accuracy): 0.3929
Performance–Fairness–Energy ranking:


,balanced_accuracy,avg_fairness_penalty,train_time_sec,srai_utility,perf_rank,fairness_rank,energy_rank,overall_rank
model,,,,,,,,
RandomForest,0.7411,0.1669,0.4040,0.6700,1.0,3.0,3.0,1.0
GradientBoosting,0.7291,0.1594,0.6573,0.6639,3.0,2.0,4.0,2.0
LogisticRegression,0.7405,0.2068,0.0660,0.6527,2.0,4.0,2.0,3.0
DummyMostFrequent,0.5000,0.0049,0.0002,0.3806,4.0,1.0,1.0,4.0


### Interpretation of the relationship between energy and fairness

1. Energy and fairness are connected through model capacity and optimization behavior. Methods with larger computational footprint can change both error rates and group disparities.
2. The relationship is not guaranteed to be monotonic. More compute does not always produce fairer outcomes, so joint evaluation is necessary.
3. A socially responsible recommendation should be based on a multi-objective criterion (`srai_utility`) rather than accuracy alone.
4. This directly addresses the instructor feedback by grounding the energy and fairness discussion in repeated experiments across seeds, methods, and datasets.

### Documentation summary for this notebook

- Datasets: Adult, COMPAS, German Credit, Titanic.
- Protected attributes: sex and race where available, plus age group in Titanic.
- Models: Dummy baseline, Logistic Regression, Random Forest, Gradient Boosting.
- Evaluation dimensions: predictive performance, fairness disparity, and compute time.
- Robustness protocol: repeated experiments over 5 random seeds (7, 21, 42, 84, 126).
- Utility function used in analysis and poster:
  - `srai_utility = 0.6*balanced_accuracy + 0.4*f1 - 0.35*avg_fairness_penalty - 0.02*log(1 + train_time_sec)`


## Reproducibility and project checklist

### How to rerun from scratch

1. Run data loading and preprocessing cells for all datasets.
2. Run baseline training and fairness evaluation cells.
3. Run expanded research cells to compute `research_results` and `summary`.
4. Run extensive comparison cells to compute `extensive_results`, method ranking, and correlations.

### Final report checklist alignment

- Problem introduction and SRAI formalization: included.
- At least two algorithms: satisfied (4 model families in extensive comparison).
- At least two evaluation methods: satisfied (multiple performance and fairness metrics).
- Data and code documentation: added in notebook narrative and reproducibility notes.
- Comprehensive comparison for bonus credit: satisfied through multi-dataset and multi-seed analysis.

### Known limitations

- Train time is used as a proxy for energy, not direct hardware power measurement.
- Hyperparameter search is intentionally lightweight to keep comparisons consistent and reproducible.
- Fairness metrics are evaluated on observed protected attributes and may not capture all real-world harms.
